# SCARLETT free-cloud render batch (piece 6)\nRuns edge-tts -> SadTalker (GPU) -> chromakey -> VP9-alpha webm for every line in a list.\n1) Runtime > Change runtime type > T4 GPU. 2) Run cells top to bottom.\n3) Optional: upload clips to your R2 bucket set in cell 3 env block.

In [ ]:
import sys, subprocess\nsubprocess.check_call([sys.executable,'-m','pip','install','-q','edge-tts','boto3'])\n!pip -q install numpy"<2" opencv-python-headless scipy librosa scikit-image scikit-learn numba tqdm kornia safetensors yacs joblib facexlib resampy imageio imageio-ffmpeg 1>/dev/null\nprint('deps ok')

In [ ]:
import urllib.request\nengine_url = 'https://raw.githubusercontent.com/abet-hq/abet-promo-engine/main/render_batch.py'\nurllib.request.urlretrieve(engine_url, '/content/render_batch.py')\nprint('engine fetched')

In [ ]:
# upload the source portrait (SCARLETT or other)\nfrom google.colab import files\nu = files.upload()\nimport os\nportrait = '/content/' + list(u.keys())[0]\nprint('portrait:', portrait)

In [ ]:
import json\nlines = [\n {'text':'Are you a loser?','name':'loser_hook'},\n {'text':'Are you a losing bettor?','name':'losing_bettor'},\n {'text':'Do you want to learn how to win, statistically?','name':'win_statistically'},\n {'text':'Do you want to beat the books at their own dirty game?','name':'beat_the_books'},\n]\njson.dump(lines, open('/content/lines.json','w'))\nprint(lines)

In [ ]:
# Optional R2 upload (leave empty to just download a zip instead)\nimport os\nfor k in ['R2_ACCOUNT_ID','R2_ACCESS_KEY_ID','R2_SECRET_ACCESS_KEY','R2_BUCKET']:\n    if k not in os.environ:\n        os.environ[k] = ''  # fill in: acct id token secret bucket\nos.environ['R2_PREFIX'] = 'clips/'\nprint('R2 set:', all(os.environ.get(k) for k in ['R2_ACCOUNT_ID','R2_BUCKET']))

In [ ]:
import subprocess, sys\nsubprocess.check_call([sys.executable,'/content/render_batch.py','--lines','/content/lines.json','--portrait',portrait,'--outdir','/content/out'])\nprint('batch done')

In [ ]:
!cd /content/out && zip -q -r /content/clips.zip .\nfrom google.colab import files\nfiles.download('/content/clips.zip')